# 00 · Preparar el entorno

## Qué vas a aprender

- Qué paquetes usa este extra y para qué sirve cada uno.
- Cuánto ocupa lo que se descarga y cuánta memoria hace falta.
- Comprobar que tu ordenador está listo **antes** de gastar tiempo en los demás notebooks.
- Qué hacer si algo falla.

**El hilo del extra.** Vamos a probar si un *LLM de visión* (un modelo que mira una foto y escribe texto) sirve para **leer el estilo de color** de una foto y escribirlo como un JSON de ajustes (temperatura, saturación, contraste...). Lo haremos en tres actos: (1) el LLM intenta leer el estilo, sin entrenar y con LoRA; (2) una regresión barata hace el trabajo; (3) el LLM intenta explicar el estilo en palabras. Lo importante no es quién gana, sino **cómo se mide** para saberlo. Este notebook solo prepara el terreno: no entrena nada.

## 1. Qué se instala y por qué

Todo va en un entorno aparte del proyecto base del curso (no engorda la imagen de la API). Se instala una vez, desde esta carpeta:

```bash
pip install -r requisitos-extra-estilo.txt
```

| Paquete | Para qué lo usamos |
|---|---|
| `torch`, `torchvision` (versión CPU) | Las redes neuronales. El LLM de visión funciona sobre `torch`. Es la misma instalación del laboratorio de red neuronal de S3 |
| `transformers` | Descarga y carga el LLM de visión (SmolVLM) |
| `peft` | LoRA: entrenar solo unos pocos millones de parámetros encima del modelo congelado (notebook 03) |
| `accelerate`, `huggingface_hub` | Piezas que `transformers` necesita para cargar modelos y descargarlos a una caché |
| `scikit-learn`, `scikit-image` | La regresión barata (Ridge), el espacio de color Lab, la diferencia de color ΔE y el SSIM |
| `numpy`, `pillow`, `matplotlib` | Cálculo con imágenes, abrirlas y dibujar los paneles |
| `mlflow` (3.18, el del curso) | Registrar los runs y el adaptador (notebooks 03 y 04) |
| `psutil` | Medir la memoria que usa el proceso |

Las **versiones exactas** están fijadas en `requisitos-extra-estilo.txt`; la sección 4 de este notebook imprime las que tienes instaladas.

**Cuánto tarda.** En el estudio previo (20/11/2026, Windows 11, CPU) instalar `transformers`, `peft`, `accelerate`, `scikit-image`, `nbconvert`, `ipykernel` y `psutil` sobre un entorno que ya tenía `torch` tardó **102 s**. La instalación de `torch` no se midió (sin verificar).

## 2. Cuánto ocupa lo que se descarga

| Qué | Para qué | Tamaño | Cuándo se descarga |
|---|---|---|---|
| SmolVLM-256M-Instruct (Hugging Face) | El LLM de visión (notebooks 02, 03 y 05) | 493,9 MB | En este notebook. Primera vez: 51,7 s en el estudio previo; depende de tu red |
| MobileNetV3-small (torchvision) | Una red pequeña ya entrenada, como línea base de comparación | 9,8 MB | En este notebook (1,3 s en el estudio previo) |
| Oxford-IIIT Pet, `images.tar.gz` | Las fotos de perros y gatos | 791,9 MB | En el notebook 01, solo si no la tienes ya |

Entre las tres filas suman unos **1,3 GB** (493,9 + 9,8 + 791,9 MB). Todo va a una carpeta de trabajo **fuera del repositorio** (no se sube a Git). Los tamaños salen del estudio previo del extra; el del modelo también lo mide la sección 5 de este notebook.

**Cuánta memoria.** En el estudio previo, el proceso llegó a **1541 MB** de memoria al preguntar al modelo (zero-shot, notebook 02) y a **2747 MB** al entrenar con LoRA (notebook 03). Esas cifras son de un portátil con 12 núcleos y 31,4 GB de RAM; el mínimo que preveíamos es un equipo de 8 GB, **sin verificar** en uno de ese tamaño.

## 3. Parámetros de este extra

La celda siguiente (la que lleva la etiqueta `parameters`) define los valores que comparten todos los notebooks. Si ejecutas el pipeline con `papermill` o con GitHub Actions, se pueden cambiar desde fuera sin tocar el notebook. Por defecto leen variables de entorno `ESTILO_*` y, si no existen, usan valores pequeños.

| Parámetro | Qué es |
|---|---|
| `N_EJEMPLOS` | Cuántas fotos se usan |
| `EPOCAS` | Cuántas pasadas de entrenamiento (notebook 03) |
| `RESOLUCION` | Lado de las fotos en píxeles |
| `MODELO` | El LLM de visión de Hugging Face |
| `CARPETA_TRABAJO` | Dónde se guardan fotos, parejas y resultados (también la variable `ESTILO_TRABAJO`) |

In [1]:
import os
N_EJEMPLOS = int(os.environ.get("ESTILO_N_EJEMPLOS", "60"))
EPOCAS = int(os.environ.get("ESTILO_EPOCAS", "2"))
RESOLUCION = int(os.environ.get("ESTILO_RESOLUCION", "224"))
MODELO = os.environ.get("ESTILO_MODELO", "HuggingFaceTB/SmolVLM-256M-Instruct")
CARPETA_TRABAJO = os.environ.get("ESTILO_TRABAJO", "_trabajo")

## 4. Qué tienes instalado

Primero localizamos el módulo `estilo` (la carpeta `estilo/` de este extra, con el código compartido: ajustes de color, métricas, carga del modelo). Después fijamos la carpeta de trabajo y los hilos de CPU, e imprimimos las versiones.

**Qué debes ver:** Python 3.11, `torch` con sufijo `+cpu` y una lista de versiones sin errores. Si falta algún paquete, el error dice cuál: instálalo con `pip install -r requisitos-extra-estilo.txt`.

In [2]:
import json, platform, sys
from importlib.metadata import version
from pathlib import Path

# Localizar la carpeta del extra (la que contiene `estilo/`), desde donde se ejecute el notebook
raiz = Path(os.environ.get('ESTILO_EXTRA') or next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'estilo').is_dir()))
sys.path.insert(0, str(raiz))
import estilo as es

trabajo = es.carpeta_trabajo(CARPETA_TRABAJO)   # se crea si no existe; fija también HF_HOME
hilos = es.limitar_hilos()
print('Carpeta de trabajo: <trabajo> (fuera del repositorio)')

entorno = {'python': platform.python_version(), 'sistema': platform.system(),
           'nucleos_logicos': os.cpu_count(), 'hilos_torch': hilos}
for paquete in ('torch', 'torchvision', 'transformers', 'peft', 'accelerate', 'huggingface_hub', 'mlflow',
                'scikit-learn', 'scikit-image', 'numpy', 'pillow', 'matplotlib'):
    entorno[paquete] = version(paquete)
for k, v in entorno.items():
    print(f'{k:16s} {v}')

Carpeta de trabajo: <trabajo> (fuera del repositorio)
python           3.11.18
sistema          Windows
nucleos_logicos  14
hilos_torch      14
torch            2.15.0+cpu
torchvision      0.29.2+cpu
transformers     5.23.0
peft             0.22.0
accelerate       1.16.0
huggingface_hub  2.4.0
mlflow           3.18.0
scikit-learn     1.9.1
scikit-image     0.26.0
numpy            2.4.6
pillow           13.0.0
matplotlib       3.11.2


## 5. Licencia y descarga del LLM de visión

No damos por supuesta la licencia: la leemos de la **ficha del modelo** en Hugging Face. Después descargamos sus archivos a la caché (`HF_HOME`). Si ya estaban, la «descarga» tarda una décima de segundo, porque no se vuelve a bajar nada: esa es la gracia de la caché.

**Qué debes ver:** una línea con el modelo, su licencia (`apache-2.0`: se puede usar, también en clase), su tamaño en MB y los segundos que tardó.

In [3]:
from huggingface_hub import model_info, snapshot_download
try:
    info = model_info(MODELO)
    entorno['licencia_modelo'] = (info.card_data or {}).get('license', 'no se pudo comprobar')
except Exception as e:
    entorno['licencia_modelo'] = f'no se pudo comprobar ({type(e).__name__})'

with es.Cronometro() as c:
    carpeta_modelo = snapshot_download(MODELO, allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja'])
entorno['s_descarga_modelo'] = round(c.s, 1)
entorno['mb_modelo_en_cache'] = es.tamano_mb(carpeta_modelo)
print(f"{MODELO}: licencia «{entorno['licencia_modelo']}»; {entorno['mb_modelo_en_cache']} MB; {c.s:.1f} s")

HuggingFaceTB/SmolVLM-256M-Instruct: licencia «apache-2.0»; 493.9 MB; 0.2 s


## 6. La red pequeña de comparación

Más adelante compararemos el LLM con una red pequeña ya entrenada, **MobileNetV3-small** (la del laboratorio de S3). Aquí solo la descargamos a la caché para que el notebook 04 no dependa de la red. Silenciamos los mensajes de descarga para que no muestren rutas de tu equipo.

Licencia: el código de `torchvision` es BSD-3-Clause; la licencia de los **pesos** entrenados con ImageNet no está verificada (sin verificar).

In [4]:
import contextlib, io
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

with es.Cronometro() as c, contextlib.redirect_stderr(io.StringIO()):
    mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
entorno['s_descarga_mobilenet'] = round(c.s, 1)
print(f'MobileNetV3-small (ImageNet): {c.s:.1f} s')

MobileNetV3-small (ImageNet): 0.2 s


## 7. ¿Cabe en tu ordenador? Memoria y disco

Miramos cuánta memoria (RAM) y cuánto disco libre tienes, y los comparamos con lo que necesitamos: unos 8 GB de RAM (el mínimo previsto, sin verificar) y espacio para los 1,3 GB de descargas (usamos 2 GB como margen).

**Qué debes ver:** tus cifras y, al final, «Todo listo». Si ves un aviso, mira la sección «Si falla» más abajo.

In [5]:
import shutil
import psutil

gb = 2 ** 30
ram_total = psutil.virtual_memory().total / gb
ram_libre = psutil.virtual_memory().available / gb
disco_libre = shutil.disk_usage(trabajo).free / gb
entorno.update(ram_total_gb=round(ram_total, 1), ram_libre_gb=round(ram_libre, 1), disco_libre_gb=round(disco_libre, 1))
print(f'Núcleos lógicos: {os.cpu_count()} | hilos de PyTorch: {hilos}')
print(f'RAM: {ram_total:.1f} GB en total, {ram_libre:.1f} GB libres')
print(f'Disco libre en la carpeta de trabajo: {disco_libre:.1f} GB')

avisos = []
if ram_total < 8:
    avisos.append('Menos de 8 GB de RAM: puede que el entrenamiento (notebook 03) no quepa.')
if disco_libre < 2:
    avisos.append('Menos de 2 GB libres en disco: puede que no quepan las descargas.')
print('\n'.join(avisos) if avisos else 'Todo listo.')

(trabajo / 'entorno.json').write_text(json.dumps(entorno, indent=1), encoding='utf-8')
print('Resumen guardado en <trabajo>/entorno.json')

Núcleos lógicos: 14 | hilos de PyTorch: 14
RAM: 31.4 GB en total, 10.5 GB libres
Disco libre en la carpeta de trabajo: 234.5 GB
Todo listo.
Resumen guardado en <trabajo>/entorno.json


## Si falla

| Síntoma | Qué probar |
|---|---|
| La descarga del modelo da error de red o se queda parada | Si tu red usa un **proxy**, define las variables `HTTPS_PROXY` y `HTTP_PROXY` antes de abrir Jupyter. Si el proxy inspecciona certificados, tu equipo de sistemas te dará el certificado de la empresa; qué variable lee cada librería (`SSL_CERT_FILE`, `REQUESTS_CA_BUNDLE`) no está verificado |
| Sigue sin bajar | Descarga el modelo desde otra red o de otro equipo y copia la carpeta de caché dentro de `HF_HOME`. Con el modelo ya en caché, ejecuta con `HF_HUB_OFFLINE=1` para que no intente conectar |
| Sale «no space left» o el disco se llena | Cambia `ESTILO_TRABAJO` a un disco con más espacio. Las fotos, el modelo y los resultados van todos a esa carpeta |
| El proceso se cierra solo, o `MemoryError` | Cierra otros programas y baja `N_EJEMPLOS`. Si aun así no cabe, sigue el extra **leyendo los notebooks ya ejecutados**: no hace falta ejecutarlos para entender el resultado |
| `ModuleNotFoundError: estilo` | El notebook no está dentro de la carpeta del extra. Ábrelo desde ahí o define la variable `ESTILO_EXTRA` con la ruta de esa carpeta |
| `licencia_modelo` dice «no se pudo comprobar» | No es grave: solo falló la consulta a la ficha. La licencia del modelo por defecto es `apache-2.0` según su ficha |

## Qué te llevas

- Sabes qué paquetes usa el extra, qué se descarga (unos 1,3 GB) y dónde queda (una carpeta fuera del repositorio).
- Comprobar el entorno antes de empezar ahorra sorpresas: licencia, memoria, disco y red.
- El modelo (493,9 MB) se descarga **una sola vez** a la caché, y los notebooks siguientes lo reutilizan.

## Para pensar

1. ¿Por qué comprobamos la licencia de un modelo en su ficha en lugar de suponerla? ¿Qué podría pasar si lo usáramos en clase sin mirarla?
2. Si en un servidor de CI (como GitHub Actions) cada trabajo descarga los 493,9 MB del modelo, ¿qué problema habría y cómo lo evitarías? (Pista: piensa en una caché compartida.)